# MLP spiral classifier — notebook companion

Notebook companion to [`../examples/01_mlp_spiral_classifier.py`](../examples/01_mlp_spiral_classifier.py). Same code, same math, cell by cell, for reading over stepping through one long file. Nothing here is simplified: every gradient is the hand-derived formula from part-2 §6-7 (backprop, shapes) and part-1 §5-7 (why depth needs a kink, softmax, cross-entropy) —

```
delta = blame
dL/dW = delta x^T
dL/db = delta
dL/dx = W^T delta
```

No autograd. A 2 → 32 → 32 → 3 ReLU network learns to separate three interleaved spiral arms that no straight line ever could — the concrete case for the "depth needs a kink" argument in part-1 §4. The `.py` file is the source of truth (gradient-checked to ~1e-8, gated behind `--check-grad`); this notebook runs the same check directly as a normal cell, since notebooks are for exploration.

In [1]:
import os

import numpy as np

SIZES = [2, 32, 32, 3]

## Data: a 3-arm spiral

A spiral is the standard "is depth+nonlinearity actually doing anything" test (CS231n). Radius grows linearly along each arm while the angle sweeps ~2 turns, offset per class so the three arms interleave. No straight line (or plane, at any depth, without a nonlinearity between layers) can separate this — a stack of linear layers with no kink between them collapses algebraically into one linear layer, so it inherits the same limitation as a single line. Only a nonlinearity (ReLU here) lets the decision boundary bend around an arm.

In [2]:
def make_spiral(n_per_class=100, n_classes=3, noise=0.2, seed=0):
    # Classic CS231n spiral: radius grows linearly along each arm while the
    # angle sweeps ~2 turns, offset per class so the arms interleave. A
    # straight line cannot separate this -- that is the whole point of the
    # kink (part-1 §5): only a nonlinearity between layers can bend a
    # decision boundary around a spiral arm.
    rng = np.random.default_rng(seed)
    X = np.zeros((n_per_class * n_classes, 2))
    y = np.zeros(n_per_class * n_classes, dtype=int)
    for c in range(n_classes):
        ix = slice(n_per_class * c, n_per_class * (c + 1))
        # A single parameter t drives both r and theta so the curve's shape
        # is independent of n_per_class (only the sampling density changes).
        # t = (i + 0.5) / n_per_class keeps t in (0, 1], i.e. r > 0 always --
        # r = 0 exactly would put every class's first point at the same
        # origin (0,0) with different labels, a contradiction, and would
        # also sit right on the ReLU's non-differentiable kink at init.
        t = (np.arange(n_per_class) + 0.5) / n_per_class
        r = t                                                  # 0 < r <= 1 along the arm
        theta = (t * 4 * np.pi                                  # ~2 turns
                 + c * (2 * np.pi / n_classes)                  # spread the arms out
                 + rng.normal(0, noise, n_per_class))           # small angular jitter
        X[ix] = np.c_[r * np.sin(theta), r * np.cos(theta)]
        y[ix] = c
    return X, y

## Init: He initialization

`std = sqrt(2/n_in)`, as in part-2 §9. Zero init is symmetric — every hidden unit would compute the same gradient forever — so the randomness is load-bearing; the scale just keeps activations sane through ReLU layers.

In [3]:
def init_params(seed=0):
    # He init: std = sqrt(2/n_in), as in part-2 §9. Zero init is symmetric --
    # every hidden unit would compute the same gradient forever -- so the
    # randomness is load-bearing, the scale just keeps activations sane.
    rng = np.random.default_rng(seed)

    def he(n_in, n_out):
        return rng.normal(0, np.sqrt(2.0 / n_in), (n_out, n_in))  # (out, in)

    params = {}
    for idx, (n_in, n_out) in enumerate(zip(SIZES[:-1], SIZES[1:]), start=1):
        params[f"W{idx}"] = he(n_in, n_out)
        params[f"b{idx}"] = np.zeros(n_out)
    return params

## Forward pass

Three linear layers, ReLU between the first two, softmax on the output. Every intermediate (`Z`, `A`) gets cached — `backward` needs `Z` for the ReLU mask and `A` as the "x" in `dL/dW = delta x^T` at each layer (part-2 §6).

In [4]:
def relu(z):
    return np.maximum(0, z)


def softmax(logits):
    # Subtract the row max before exponentiating (part-1 §6): shifts every
    # logit by a constant, which cancels in the ratio, and stops exp()
    # overflowing on large inputs.
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(X, params):
    # Save every intermediate -- backward needs Z (for the ReLU mask) and
    # A (as the "x" in dL/dW = delta x^T) at each layer (part-2 §6).
    Z1 = X @ params["W1"].T + params["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ params["W2"].T + params["b2"]
    A2 = relu(Z2)
    Z3 = A2 @ params["W3"].T + params["b3"]
    probs = softmax(Z3)
    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "probs": probs}
    return Z3, cache

## Loss: softmax cross-entropy

`probs[arange(B), y]` reads out the predicted probability of the true class for all `B` examples at once (part-1 §7). `+1e-12` just guards against `log(0)`.

In [5]:
def cross_entropy_loss(probs, y):
    B = y.shape[0]
    # probs[arange(B), y]: the predicted probability of the true class,
    # read out for all B examples at once (part-1 §7).
    return -np.mean(np.log(probs[np.arange(B), y] + 1e-12))  # +eps: never log(0)

## Backward pass: hand-derived chain rule

Backprop is bookkeeping applied three times, once per layer, all using the same two formulas from part-2 §6: `dL/dW = delta x^T` and `dL/db = delta`, where `delta` is "how much blame this layer's output deserves."

The chain starts where softmax and cross-entropy meet: their gradients collapse algebraically to exactly `predicted - actual` — that's `delta3`, the blame at the output layer. From there each earlier layer's blame is the next layer's blame projected back through that layer's weights (`dL/dx = W^T delta`), then masked by the ReLU: a unit that was off (`Z <= 0`) during the forward pass contributed nothing, so it gets no blame. Layout note: `delta` here is a row per example, so the projection is written `delta @ W` rather than `W.T @ delta` — same formula, transposed layout for row-major batches.

No autograd, no numerical differentiation in this function — those are for the standalone check further down.

In [6]:
def backward(params, cache, y):
    # Hand-derived chain rule only -- no autograd, no numerical gradients
    # here (those are for the standalone check, see check_gradients below).
    B = y.shape[0]
    Y = np.zeros_like(cache["probs"])
    Y[np.arange(B), y] = 1  # one-hot true class per row

    # softmax + cross-entropy collapse to exactly (predicted - actual),
    # the delta ("blame") for the output layer (part-1 §7, part-2 §6).
    delta3 = (cache["probs"] - Y) / B  # /B averages the loss over the batch

    dW3 = delta3.T @ cache["A2"]       # dL/dW = delta x^T
    db3 = delta3.sum(axis=0)           # dL/db = delta

    # dL/dx = W^T delta, then the ReLU mask blocks blame at units that were
    # off during the forward pass (they contributed nothing, so they get
    # no blame). delta is a row per example here, so it's `@ W` rather than
    # `W.T @ delta` -- same formula, transposed layout.
    delta2 = (delta3 @ params["W3"]) * (cache["Z2"] > 0)
    dW2 = delta2.T @ cache["A1"]
    db2 = delta2.sum(axis=0)

    delta1 = (delta2 @ params["W2"]) * (cache["Z1"] > 0)
    dW1 = delta1.T @ cache["X"]
    db1 = delta1.sum(axis=0)

    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}

## Train: plain full-batch gradient descent

No optimizer tricks — just `param -= lr * grad`, every epoch, on the whole dataset at once.

In [7]:
def train(epochs=4000, lr=0.5, n_per_class=100, n_classes=3, noise=0.2,
          seed=0, print_every=250, save_path=None):
    X, y = make_spiral(n_per_class, n_classes, noise, seed)
    params = init_params(seed)

    for epoch in range(epochs):
        logits, cache = forward(X, params)
        grads = backward(params, cache, y)
        for k in params:
            params[k] -= lr * grads[k]  # plain full-batch gradient descent

        if epoch % print_every == 0 or epoch == epochs - 1:
            loss = cross_entropy_loss(cache["probs"], y)
            acc = (cache["probs"].argmax(axis=1) == y).mean()
            print(f"epoch {epoch:4d}   loss {loss:.4f}   train acc {acc:.4f}")

    if save_path is None:
        save_path = os.path.join(os.path.abspath("."), "01_weights.npz")
    np.savez(save_path, **params)
    print(f"saved weights to {save_path}")

    return params

## Predict

Loads params (dict or `.npz` path), runs `forward`, returns the argmax class and the full probability vector.

In [8]:
def predict(X, params_or_path):
    if isinstance(params_or_path, (str, os.PathLike)):
        with np.load(params_or_path) as data:
            params = {k: data[k] for k in data.files}
    else:
        params = params_or_path

    logits, cache = forward(X, params)
    probs = cache["probs"]
    return probs.argmax(axis=1), probs

## Run: train the network

Same hyperparameters as the `.py` file's `__main__` block: 4000 epochs, `lr=0.5`, printing every 250 epochs.

In [9]:
params = train(epochs=4000, lr=0.5, print_every=250)

epoch    0   loss 1.2036   train acc 0.3233
epoch  250   loss 1.0203   train acc 0.4733
epoch  500   loss 0.9784   train acc 0.4633


epoch  750   loss 0.9409   train acc 0.5033
epoch 1000   loss 0.8583   train acc 0.5867
epoch 1250   loss 0.7407   train acc 0.6533


epoch 1500   loss 0.5885   train acc 0.7300
epoch 1750   loss 0.5338   train acc 0.8067
epoch 2000   loss 0.3106   train acc 0.8967


epoch 2250   loss 0.2766   train acc 0.8867
epoch 2500   loss 0.1263   train acc 0.9633
epoch 2750   loss 0.5271   train acc 0.8133


epoch 3000   loss 0.0529   train acc 0.9900
epoch 3250   loss 0.0369   train acc 0.9967
epoch 3500   loss 0.0290   train acc 0.9967


epoch 3750   loss 0.0250   train acc 0.9967
epoch 3999   loss 0.0198   train acc 0.9933
saved weights to /private/tmp/claude-501/-Users-ved-workplace-ai-genai/ab231e17-f0d5-45d5-8d97-53421764dd25/scratchpad/01_weights.npz


## Verify: numerical gradient check

Central-difference check of `backward()` against `forward()` alone — the same idea as part-2's "verify with a tiny numerical check" sidebar. This is *not* how the network trains; it only proves the hand-derived formulas above are correct. In the `.py` file this is gated behind `--check-grad`; here it just runs as a normal cell.

In [10]:
def check_gradients(seed=1, n_checks=3, eps=1e-5):
    # Central-difference check of backward() against forward() alone --
    # the same idea as part-2's "verify with a tiny numerical check"
    # sidebar. This is *not* how the network trains; it only proves the
    # hand-derived formulas above are correct.
    rng = np.random.default_rng(seed)
    X, y = make_spiral(n_per_class=5, n_classes=3, noise=0.2, seed=seed)
    params = init_params(seed)
    _, cache = forward(X, params)
    grads = backward(params, cache, y)

    def loss_at(params):
        _, cache = forward(X, params)
        return cross_entropy_loss(cache["probs"], y)

    worst_rel_err = 0.0
    for name in params:
        shape = params[name].shape
        idxs = [tuple(rng.integers(0, d) for d in shape) for _ in range(n_checks)]
        for idx in idxs:
            orig = params[name][idx]

            params[name][idx] = orig + eps
            loss_plus = loss_at(params)
            params[name][idx] = orig - eps
            loss_minus = loss_at(params)
            params[name][idx] = orig  # restore

            numeric = (loss_plus - loss_minus) / (2 * eps)
            analytic = grads[name][idx]
            rel_err = abs(numeric - analytic) / max(abs(numeric), abs(analytic), 1e-8)
            worst_rel_err = max(worst_rel_err, rel_err)
            print(f"{name}{idx}: analytic {analytic:+.6e}  numeric {numeric:+.6e}  rel_err {rel_err:.2e}")

    print(f"worst relative error: {worst_rel_err:.2e}")
    return worst_rel_err


err = check_gradients()
print("PASS" if err < 1e-6 else "FAIL", "(threshold 1e-6)")

W1(np.int64(15), np.int64(1)): analytic -3.379817e-03  numeric -3.379817e-03  rel_err 1.41e-09
W1(np.int64(24), np.int64(1)): analytic -7.096041e-05  numeric -7.096040e-05  rel_err 6.42e-08
W1(np.int64(1), np.int64(0)): analytic -3.488117e-03  numeric -3.488117e-03  rel_err 5.52e-10
b1(np.int64(26),): analytic -1.732406e-02  numeric -1.732406e-02  rel_err 5.47e-11
b1(np.int64(30),): analytic -1.178261e-02  numeric -1.178261e-02  rel_err 6.56e-10
b1(np.int64(7),): analytic +8.972722e-03  numeric +8.972722e-03  rel_err 2.60e-10
W2(np.int64(9), np.int64(27)): analytic -1.399893e-03  numeric -1.399893e-03  rel_err 3.81e-09
W2(np.int64(13), np.int64(8)): analytic -4.978845e-03  numeric -4.978845e-03  rel_err 2.21e-10
W2(np.int64(26), np.int64(8)): analytic +8.034077e-04  numeric +8.034077e-04  rel_err 1.26e-08
b2(np.int64(13),): analytic -3.358048e-02  numeric -3.358048e-02  rel_err 5.15e-10
b2(np.int64(20),): analytic +1.621560e-02  numeric +1.621560e-02  rel_err 1.72e-10
b2(np.int64(17),)

## Inference demo

First, accuracy on the training spiral itself. Then predictions on a handful of new points (seed=99, never seen during training) — matching the `.py` file's `__main__` demo.

In [11]:
X, y = make_spiral(n_per_class=100, n_classes=3, noise=0.2, seed=0)
preds, _ = predict(X, params)
acc = (preds == y).mean()
print(f"final train accuracy: {acc:.4f}")

# A handful of new points (seed=99), never seen during training.
X_new, y_new = make_spiral(n_per_class=5, n_classes=3, noise=0.2, seed=99)
preds_new, probs_new = predict(X_new, params)
print("\nsample predictions on new points:")
for i in range(len(X_new)):
    confidence = probs_new[i, preds_new[i]]
    mark = "ok" if preds_new[i] == y_new[i] else "MISS"
    print(f"  x={X_new[i]}  true={y_new[i]}  pred={preds_new[i]}  "
          f"confidence={confidence:.3f}  [{mark}]")

final train accuracy: 0.9967

sample predictions on new points:
  x=[0.09560253 0.02932843]  true=0  pred=0  confidence=0.997  [ok]
  x=[-0.15306452 -0.25801405]  true=0  pred=0  confidence=0.965  [ok]
  x=[0.00505142 0.49997448]  true=0  pred=0  confidence=1.000  [ok]
  x=[ 0.33010026 -0.61727937]  true=0  pred=0  confidence=0.995  [ok]
  x=[-0.89937738 -0.03347123]  true=0  pred=0  confidence=1.000  [ok]
  x=[-0.05195514 -0.08544392]  true=1  pred=1  confidence=1.000  [ok]
  x=[-0.14657035  0.26175778]  true=1  pred=1  confidence=0.997  [ok]
  x=[ 0.45968609 -0.19669443]  true=1  pred=1  confidence=0.620  [ok]
  x=[-0.66574951 -0.21628127]  true=1  pred=1  confidence=0.997  [ok]
  x=[0.76882968 0.46786848]  true=1  pred=1  confidence=0.940  [ok]
  x=[-0.06463277  0.076306  ]  true=2  pred=2  confidence=1.000  [ok]
  x=[ 0.28143883 -0.10388545]  true=2  pred=2  confidence=0.516  [ok]
  x=[-0.47054089 -0.16908954]  true=2  pred=2  confidence=0.999  [ok]
  x=[0.31794372 0.62362793]  tru